## Setup

In [17]:
import sys
import os
import importlib

from dotenv import load_dotenv

sys.path.append("../src")

import utils
import metrics

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(metrics)

# NOTE: define variables in .env to avoid changing the notebook
load_dotenv("../.env", override=True)

True

In [18]:
FILENAME = os.getenv("FILENAME", "parkinson")
BACKEND = os.getenv("BACKEND", "openai")
CUMULATIVE = os.getenv("CUMULATIVE", "True").lower() == "true"
DST = f"../results/{BACKEND}/{FILENAME}"

print(f"{FILENAME=}")
print(f"{BACKEND=}")
print(f"{CUMULATIVE=}")

df = utils.load(f"../data/embeddings/{BACKEND}/{FILENAME}.csv")

print(f"{df.shape=}")
df.head(3)

FILENAME='german'
BACKEND='openai'
CUMULATIVE=True
df.shape=(10010, 9)


,num,id,concept_en,category,concept,property,prop_embedding,properties_cum,embedding
0,1,s1,broom,implement,Besen,damit kehren,"[tensor(-0.0293), tensor(-0.0049), tensor(-0.0...",damit kehren,"[tensor(-0.0293), tensor(-0.0049), tensor(-0.0..."
1,2,s1,broom,implement,Besen,hat Bürste,"[tensor(-0.0156), tensor(-0.0016), tensor(-0.0...",damit kehren hat Bürste,"[tensor(-0.0308), tensor(-0.0125), tensor(-0.0..."
2,3,s1,broom,implement,Besen,hat Stiel,"[tensor(-0.0109), tensor(-0.0211), tensor(-0.0...",damit kehren hat Bürste hat Stiel,"[tensor(-0.0374), tensor(-0.0059), tensor(-0.0..."


In [19]:
# NOTE: enable ZCA whitening
# df, zca_params = utils.zca_whitened_embeddings(df, emb_col="embedding", new_col="embedding_zca")
# df = df.drop(columns=["embedding"]).rename(columns={"embedding_zca": "embedding"})
# df.head(3)

## Metrics

In [20]:
# TODO: understand dataframe "management/updates" here
# grouped = df.groupby(["ID", "Concept"], group_keys=False)

In [ ]:
# TODO: mention what's being computed here
df = (
    df.groupby(["id", "concept"], group_keys=False)
    .apply(
        metrics.geometric,
        include_groups=True,
        cumulative=CUMULATIVE,
    )
    .reset_index(drop=True)
)

/var/folders/2t/jw0zh_ts0q19czmdc_ssfz2h0000gn/T/ipykernel_82923/127948418.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  .apply(
/var/folders/2t/jw0zh_ts0q19czmdc_ssfz2h0000gn/T/ipykernel_82923/127948418.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [ ]:
# TODO: mention what's being computed here
df = (
    df.groupby(["id", "concept"], group_keys=True)
    .apply(
        metrics.kinematic,
        # include_groups=True
        cumulative=CUMULATIVE,
    )
    .reset_index(drop=True)
)

/var/folders/2t/jw0zh_ts0q19czmdc_ssfz2h0000gn/T/ipykernel_82923/1686520772.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [23]:
# Keep only original columns + computed metrics (not embeddings)
to_drop = ("prop_embedding", "properties_cum", "embedding", "vel_vector", "acc_vector")
cols = [col for col in df.columns if col not in to_drop]
df = df[cols]

metrics = ["entropy", "d_next", "d_centroid", "vel", "acc"]
print(f"NaN values: \n\n{df[metrics].isna().mean()}\n")

# Inspect sample of results
df.head(3)

NaN values: 

entropy       0.208392
d_next        0.182218
d_centroid    0.000000
vel           0.182218
acc           0.364036
dtype: float64



,num,id,concept_en,category,concept,property,d_next,entropy,d_centroid,vel,acc
0,42,s1,monkey,mammal,Affe,sehr menschenähnlich,0.285317,NaN,0.261954,0.755402,0.971441
1,43,s1,monkey,mammal,Affe,lebt im Dschungel,0.130834,NaN,0.282684,0.511534,0.645531
2,44,s1,monkey,mammal,Affe,kann sehr groß sein,0.048172,NaN,0.237616,0.310394,<NA>


In [24]:
# Save results
os.makedirs(DST, exist_ok=True)
df[metrics].isna().mean().to_csv(f"{DST}/metrics-nan.csv")
utils.save(df, f"{DST}/metrics.csv")
print(f"{df.shape=}")

df.shape=(10010, 11)
